# Encoder vs decoder comparison (smoke eval sample)

Joins `encoder_smoke_scores.parquet` + `decoder_smoke_scores.parquet` on `comment_text`.

**Prerequisites:** run notebook 02 §13 (decoder parquet). Then either run notebook 01 save cell or:
```bash
python scripts/score_encoder_smoke.py --from-decoder
python scripts/join_scores.py
```

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

THRESHOLD = 0.5

_cwd = Path.cwd()
_bootstrap = _cwd.parent if _cwd.name == "notebooks" else _cwd
_src = str((_bootstrap / "src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

for _mod in ("paths", "evaluate", "plots", "decoder", "join_scores"):
    sys.modules.pop(_mod, None)

from decoder import format_method_summary_text, d1_method_summary
from evaluate import agreement_by_label
from paths import (
    DECODER_SMOKE_SCORES_PATH,
    ENCODER_SMOKE_SCORES_PATH,
    JOINED_SCORES_PATH,
    TOXIC_LABELS,
)
from plots import (
    metrics_from_joined,
    plot_agreement_by_label,
    plot_gold_agreement_panel,
    plot_macro_f1_comparison,
    plot_prob_scatter,
)

sns.set_theme(style="whitegrid")

## 1. Load joined scores

In [ ]:
if JOINED_SCORES_PATH.exists():
    joined = pd.read_parquet(JOINED_SCORES_PATH)
    print(f"Loaded {len(joined)} rows from {JOINED_SCORES_PATH}")
elif DECODER_SMOKE_SCORES_PATH.exists() and ENCODER_SMOKE_SCORES_PATH.exists():
    from join_scores import join_encoder_decoder
    enc = pd.read_parquet(ENCODER_SMOKE_SCORES_PATH)
    dec = pd.read_parquet(DECODER_SMOKE_SCORES_PATH)
    joined = join_encoder_decoder(enc, dec)
    print(f"Joined in-memory: {len(joined)} rows")
else:
    raise FileNotFoundError(
        "Missing parquets. Run nb02 §13, then: python scripts/score_encoder_smoke.py --from-decoder"
    )

display(joined[["comment_text", "toxic"] + [f"enc_pred_toxic", "d1_pred_logprob_toxic"]].head())

## 2. Agreement: encoder vs D1 logprob

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
_, agree_df = plot_agreement_by_label(joined, ax=ax)
display(agree_df.round(3))
mean_agree = agree_df["agreement"].mean()
print(
    f"Encoder vs D1 logprob on {len(joined)} comments — mean agreement {mean_agree:.0%} "
    f"(best: {agree_df.loc[agree_df['agreement'].idxmax(), 'label']} "
    f"{agree_df['agreement'].max():.0%})."
)
plt.show()

## 3. Each method vs gold

In [ ]:
fig, ax, gold_agree = plot_gold_agreement_panel(joined)
display(gold_agree.round(3))
enc_mean = gold_agree.loc[gold_agree["method"] == "encoder", "agreement"].mean()
dec_mean = gold_agree.loc[gold_agree["method"] == "decoder D1", "agreement"].mean()
winner = "encoder" if enc_mean >= dec_mean else "decoder D1"
print(
    f"On {len(joined)} balanced comments, {winner} matches gold more often on average "
    f"(encoder {enc_mean:.0%} vs decoder {dec_mean:.0%})."
)
plt.show()

## 4. Formal multilabel metrics (F1 / PR-AUC)

In [ ]:
enc_m = metrics_from_joined(joined, method="encoder", threshold=THRESHOLD)
dec_m = metrics_from_joined(joined, method="decoder", threshold=THRESHOLD)

print("Encoder summary:", {k: round(v, 3) for k, v in enc_m["summary"].items()})
print("Decoder D1 summary:", {k: round(v, 3) for k, v in dec_m["summary"].items()})

fig, ax, _ = plot_macro_f1_comparison(enc_m, dec_m, split_name="smoke eval")
plt.show()

## 5. Probability scatter (toxic)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_prob_scatter(joined, label="toxic", ax=ax)
plt.show()

## 6. D1 method breakdown (decoder only)

In [ ]:
d1_summary = d1_method_summary(joined)
print(format_method_summary_text(d1_summary, len(joined), prompt_name="D1 on joined sample"))
display(d1_summary.round(3))